# Ridge Regression

Amarasekara I.S.Y. — IT25101702

Used-car price prediction

## Load the data

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load training, validation and test data
tuning_train = pd.read_csv('tuning_train.csv')
validation = pd.read_csv('tuning_validation.csv')
train = pd.read_csv('processed_train.csv')
test = pd.read_csv('processed_holdout.csv')

# Check the number of rows and columns
print(tuning_train.shape)
print(validation.shape)
print(train.shape)
print(test.shape)

from sklearn.linear_model import Ridge

## Separate inputs and prices

In [ ]:
# Separate features and targets for tuning
X_train = tuning_train.drop(columns=['Price', 'Log_Price'])
y_train = tuning_train['Price']
y_train_log = tuning_train['Log_Price']

# Prepare the validation set
X_val = validation.drop(columns=['Price', 'Log_Price'])
y_val = validation['Price']

# Prepare the full training set for the selected model
X_final = train.drop(columns=['Price', 'Log_Price'])
y_final = train['Price']
y_final_log = train['Log_Price']

# Prepare the labelled test set
X_test = test.drop(columns=['Price', 'Log_Price'])
y_test = test['Price']

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

## Train the two baseline variants

In [ ]:
# Train with default settings using prices
model1 = Ridge()
model1.fit(X_train, y_train)

# Predict validation prices
prediction1 = model1.predict(X_val)

# Calculate validation scores
rmse1 = np.sqrt(mean_squared_error(y_val, prediction1))
mae1 = mean_absolute_error(y_val, prediction1)
r2_1 = r2_score(y_val, prediction1)
print('Default + Price:', rmse1, mae1, r2_1)

# Train with default settings using log prices
model2 = Ridge()
model2.fit(X_train, y_train_log)

# Predict validation prices and reverse the log transformation
prediction2 = np.expm1(model2.predict(X_val))

# Calculate validation scores
rmse2 = np.sqrt(mean_squared_error(y_val, prediction2))
mae2 = mean_absolute_error(y_val, prediction2)
r2_2 = r2_score(y_val, prediction2)
print('Default + Log_Price:', rmse2, mae2, r2_2)

## Try changed settings and compare

In [ ]:
# Train with changed settings using prices
model3 = Ridge(alpha=10)
model3.fit(X_train, y_train)

# Predict validation prices
prediction3 = model3.predict(X_val)

# Calculate validation scores
rmse3 = np.sqrt(mean_squared_error(y_val, prediction3))
mae3 = mean_absolute_error(y_val, prediction3)
r2_3 = r2_score(y_val, prediction3)
print('Changed settings + Price:', rmse3, mae3, r2_3)

# Train with changed settings using log prices
model4 = Ridge(alpha=10)
model4.fit(X_train, y_train_log)

# Predict validation prices and reverse the log transformation
prediction4 = np.expm1(model4.predict(X_val))

# Calculate validation scores
rmse4 = np.sqrt(mean_squared_error(y_val, prediction4))
mae4 = mean_absolute_error(y_val, prediction4)
r2_4 = r2_score(y_val, prediction4)
print('Changed settings + Log_Price:', rmse4, mae4, r2_4)

# Compare the four variants
comparison = pd.DataFrame({
    'Variant': ['Default + Price', 'Default + Log_Price',
                'Changed settings + Price', 'Changed settings + Log_Price'],
    'RMSE': [rmse1, rmse2, rmse3, rmse4],
    'MAE': [mae1, mae2, mae3, mae4],
    'R2': [r2_1, r2_2, r2_3, r2_4]
})
print(comparison.sort_values('RMSE'))
comparison.to_csv('variant_comparison.csv', index=False)

# Choose the variant with the lowest validation RMSE
models = [model1, model2, model3, model4]
best_index = comparison['RMSE'].idxmin()
print('Selected variant:', comparison.loc[best_index, 'Variant'])

## Evaluate on the labelled test set

In [ ]:
# Select the model with the lowest validation RMSE
best_model = models[best_index]

# Retrain on the full training set and predict test prices
# Indices 1 and 3 are the log-price variants
if best_index in [1, 3]:
    best_model.fit(X_final, y_final_log)
    test_prediction = np.expm1(best_model.predict(X_test))
    train_prediction = np.expm1(best_model.predict(X_final))
else:
    best_model.fit(X_final, y_final)
    test_prediction = best_model.predict(X_test)
    train_prediction = best_model.predict(X_final)

# Calculate scores in the original price units
rmse = np.sqrt(mean_squared_error(y_test, test_prediction))
mae = mean_absolute_error(y_test, test_prediction)
r2 = r2_score(y_test, test_prediction)
train_rmse = np.sqrt(mean_squared_error(y_final, train_prediction))

print('Training RMSE:', train_rmse)
print('Test RMSE:', rmse)
print('Test MAE:', mae)
print('Test R2:', r2)

# Plot actual prices against predicted prices
plt.figure()
plt.scatter(y_test, test_prediction, alpha=0.4)
plt.plot([0, y_test.max()], [0, y_test.max()], 'r--')
plt.xlabel('Actual price (INR lakhs)')
plt.ylabel('Predicted price (INR lakhs)')
plt.title('Actual and predicted prices')
plt.savefig('prediction_plot.png', dpi=150)
plt.show()

# Plot residuals to check the prediction errors
plt.figure()
plt.scatter(test_prediction, y_test - test_prediction, alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted price (INR lakhs)')
plt.ylabel('Actual minus predicted price (INR lakhs)')
plt.title('Prediction errors')
plt.savefig('residual_plot.png', dpi=150)
plt.show()

## Save results and make predictions

In [ ]:
# Save the selected variant and evaluation scores
result = pd.DataFrame({
    'Model': ['Ridge Regression'],
    'Model_ID': ['ridge'],
    'Student_ID': ['IT25101702'],
    'Protocol': ['cars12-v3-validation42'],
    'Variant': [comparison.loc[best_index, 'Variant']],
    'Validation_RMSE': [comparison.loc[best_index, 'RMSE']],
    'Validation_MAE': [comparison.loc[best_index, 'MAE']],
    'Validation_R2': [comparison.loc[best_index, 'R2']],
    'Train_RMSE': [train_rmse],
    'Holdout_RMSE': [rmse], 'Holdout_MAE': [mae], 'Holdout_R2': [r2],
    'Train_Rows': [len(X_final)], 'Holdout_Rows': [len(X_test)],
    'Tuning_Rows': [len(X_train)], 'Validation_Rows': [len(X_val)],
    'Features': [X_final.shape[1]]
})
result.to_csv('ridge_result.csv', index=False)

# Predict prices for the unlabelled university data
university = pd.read_csv('processed_university_test.csv')
university_prediction = best_model.predict(university)
if best_index in [1, 3]:
    university_prediction = np.expm1(university_prediction)

# Keep predictions in the same order as the input rows
predictions = pd.DataFrame({
    'Row_Number': range(1, len(university) + 1),
    'Predicted_Price_INR_Lakhs': university_prediction
})
predictions.to_csv('ridge_university_predictions.csv', index=False)
print(result.T)
print(predictions.head())
print(predictions.shape)